# Baseline Model Training — RandomForest Classifier

**Project:** E-commerce Sales Data Analysis & Machine Learning  
**Author:** Muhammad Iqbal Fadel  
**Source:** `scripts/04_baseline_train.py`

## Overview

Train a baseline RandomForest classifier for customer segmentation.

Includes:
- Stratified train/test split
- Cross-validation (5-fold)
- Feature importance visualization
- Confusion matrix visualization
- ROC curve
- Full classification report

Author: Muhammad Iqbal Fadel
Date: May 2026 (updated June 2026)

In [ ]:
%matplotlib inline

import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import joblib

sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Paths ──────────────────────────────────────────────────────────────────

In [ ]:
ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'model_data_customers.csv')
FIG_DIR = os.path.join(ROOT, 'outputs', 'figures', 'models')
DATA_DIR = os.path.join(ROOT, 'outputs', 'data', 'models')

for d in (FIG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

## 1. LOAD DATA

In [ ]:
print('=' * 60)
print('  BASELINE MODEL TRAINING — RandomForest Classifier')
print('=' * 60)

print(f'\nLoading {DATA_PATH}')
df = pd.read_csv(DATA_PATH)

features = ['DistinctProducts', 'Frequency', 'Recency', 'Monetary', 'AvgOrderValue']
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['Label']

print(f'  Dataset: {len(df)} customers')
print(f'  Features: {features}')
print(f'  Target distribution: {dict(y.value_counts())}')

## 2. TRAIN / TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\n  Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

## 3. TRAIN MODEL

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

## 4. EVALUATION

In [ ]:
print('\n' + '=' * 60)
print('  EVALUATION RESULTS')
print('=' * 60)

# Classification Report
report_str = classification_report(y_test, y_pred)
print('\nClassification Report:')
print(report_str)

# ROC AUC
roc = roc_auc_score(y_test, y_proba)
print(f'ROC AUC: {roc:.4f}')

## 5. CROSS-VALIDATION (5-Fold)

In [ ]:
print('\n' + '=' * 60)
print('  CROSS-VALIDATION (5-Fold Stratified)')
print('=' * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_acc = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
cv_scores_roc = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')

print(f'\n  Accuracy:  {cv_scores_acc.mean():.4f} ± {cv_scores_acc.std():.4f}')
print(f'  ROC AUC:   {cv_scores_roc.mean():.4f} ± {cv_scores_roc.std():.4f}')
print(f'  Per-fold accuracy: {[f"{s:.4f}" for s in cv_scores_acc]}')
print(f'  Per-fold ROC AUC:  {[f"{s:.4f}" for s in cv_scores_roc]}')

## 6. SAVE MODEL & METRICS

In [ ]:
model_path = os.path.join(DATA_DIR, 'baseline_rf.pkl')
joblib.dump(clf, model_path)

metrics = {
    'roc_auc': float(roc),
    'cv_accuracy_mean': float(cv_scores_acc.mean()),
    'cv_accuracy_std': float(cv_scores_acc.std()),
    'cv_roc_auc_mean': float(cv_scores_roc.mean()),
    'cv_roc_auc_std': float(cv_scores_roc.std()),
    'n_features': len(features),
    'features': features,
}
with open(os.path.join(DATA_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n  [OK] Model saved: {model_path}')
print(f'  [OK] Metrics saved: {os.path.join(DATA_DIR, "baseline_metrics.json")}')

## 7. VISUALIZATIONS

In [ ]:
print('\n' + '=' * 60)
print('  GENERATING VISUALIZATIONS')
print('=' * 60)

## ── 7a. ROC Curve ────────────────────────────────────────────────────────

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#2196F3', linewidth=2.5,
        label=f'RandomForest (AUC = {roc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
ax.fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Baseline RandomForest')
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3)
roc_path = os.path.join(FIG_DIR, 'baseline_roc.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'  [OK] Saved: {roc_path}')

## ── 7b. Feature Importance ───────────────────────────────────────────────

In [ ]:
importances = clf.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_imp['Feature'], feat_imp['Importance'],
               color='#4CAF50', edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, feat_imp['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('RandomForest — Feature Importance')
ax.grid(axis='x', alpha=0.3)
fi_path = os.path.join(FIG_DIR, 'feature_importance.png')
plt.savefig(fi_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'  [OK] Saved: {fi_path}')

## ── 7c. Confusion Matrix ────────────────────────────────────────────────

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Champion (0)', 'Champion (1)'],
            yticklabels=['Non-Champion (0)', 'Champion (1)'],
            linewidths=1, linecolor='white', ax=ax,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — Baseline RandomForest')
cm_path = os.path.join(FIG_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'  [OK] Saved: {cm_path}')

## ── 7d. Cross-Validation Scores Bar Chart ────────────────────────────────

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fold_labels = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
width = 0.35
bars1 = ax.bar(x - width/2, cv_scores_acc, width, label='Accuracy',
               color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, cv_scores_roc, width, label='ROC AUC',
               color='#FF5722', edgecolor='white')
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.set_ylabel('Score')
ax.set_title('5-Fold Cross-Validation Scores')
ax.set_ylim(0.75, 1.02)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
cv_path = os.path.join(FIG_DIR, 'cross_validation_scores.png')
plt.savefig(cv_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f'  [OK] Saved: {cv_path}')

## ── Summary ──────────────────────────────────────────────────────────────

In [ ]:
print('\n' + '=' * 60)
print('  TRAINING COMPLETE')
print('=' * 60)
print(f'  Model:       RandomForest (200 trees)')
print(f'  ROC AUC:     {roc:.4f}')
print(f'  CV Accuracy: {cv_scores_acc.mean():.4f} ± {cv_scores_acc.std():.4f}')
print(f'  CV ROC AUC:  {cv_scores_roc.mean():.4f} ± {cv_scores_roc.std():.4f}')
print(f'  Figures:     {FIG_DIR}')
print(f'  Data:        {DATA_DIR}')